# Scene Object / Skeleton / Gaze Viewer

Interactive Plotly viewer for ADT Scene-frame context. It reads precomputed CSVs from `REPORTS_DIR`:

- `*_scene_object_boxes.csv`
- `*_skeleton_samples.csv`
- `*_head_samples.csv`
- `*_gaze_samples.csv`

The first version is for coordinate-frame validation: object boxes, skeleton joints, head/device trajectory, gaze rays, and depth-defined gaze points should all appear in the same Scene frame.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from adt_sandbox.scene_object_viewer import (
    build_scene_object_gaze_figure,
    discover_scene_viewer_sequence_ids,
    load_scene_viewer_data,
)

REPORTS_DIR = Path("/mnt/d/SparseGaze/ADT-Gaze-structured")
sequence_ids = discover_scene_viewer_sequence_ids(REPORTS_DIR)
sequence_ids[:3], len(sequence_ids)

(['Apartment_release_decoration_skeleton_seq131_M1292',
  'Apartment_release_decoration_skeleton_seq132_M1292',
  'Apartment_release_decoration_skeleton_seq133_M1292'],
 34)

In [2]:
import ipywidgets as widgets
from IPython.display import display

_data_cache = {}

def get_data(sequence_id):
    if sequence_id not in _data_cache:
        _data_cache[sequence_id] = load_scene_viewer_data(REPORTS_DIR, sequence_id)
    return _data_cache[sequence_id]

sequence_dropdown = widgets.Dropdown(
    options=sequence_ids,
    value=sequence_ids[0],
    description="sequence",
    layout=widgets.Layout(width="900px"),
)
start_input = widgets.IntText(value=0, description="start")
end_input = widgets.IntText(value=180, description="end")
stride_input = widgets.IntText(value=10, description="stride")
focus_input = widgets.IntText(value=-1, description="focus")
max_static_input = widgets.IntText(value=120, description="max static")
category_text = widgets.Text(value="", description="categories", layout=widgets.Layout(width="520px"))

show_static_checkbox = widgets.Checkbox(value=True, description="static boxes")
show_dynamic_checkbox = widgets.Checkbox(value=True, description="dynamic boxes")
show_centers_checkbox = widgets.Checkbox(value=False, description="object centers")
show_skeleton_checkbox = widgets.Checkbox(value=True, description="skeleton")
show_skeleton_traj_checkbox = widgets.Checkbox(value=True, description="root path")
show_head_checkbox = widgets.Checkbox(value=True, description="head path")
show_gaze_rays_checkbox = widgets.Checkbox(value=True, description="gaze rays")
show_gaze_points_checkbox = widgets.Checkbox(value=True, description="gaze points")

gaze_length_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="ray len",
    readout_format=".1f",
)
gaze_scale_dropdown = widgets.Dropdown(
    options=["fixed", "depth"],
    value="fixed",
    description="ray scale",
)
render_button = widgets.Button(description="Render", button_style="primary")
output = widgets.Output()

def render(_=None):
    with output:
        output.clear_output(wait=True)
        data = get_data(sequence_dropdown.value)
        focus_frame = None if focus_input.value < 0 else focus_input.value
        max_static = None if max_static_input.value <= 0 else max_static_input.value
        fig = build_scene_object_gaze_figure(
            data,
            start_frame=start_input.value,
            end_frame=end_input.value,
            stride=stride_input.value,
            focus_frame=focus_frame,
            show_static_objects=show_static_checkbox.value,
            show_dynamic_objects=show_dynamic_checkbox.value,
            show_object_centers=show_centers_checkbox.value,
            max_static_objects=max_static,
            category_filter=category_text.value,
            show_skeleton=show_skeleton_checkbox.value,
            show_skeleton_trajectory=show_skeleton_traj_checkbox.value,
            show_head_trajectory=show_head_checkbox.value,
            show_gaze_rays=show_gaze_rays_checkbox.value,
            show_gaze_points=show_gaze_points_checkbox.value,
            gaze_ray_length_m=gaze_length_slider.value,
            gaze_scale_mode=gaze_scale_dropdown.value,
        )
        display(fig)

render_button.on_click(render)

controls = widgets.VBox([
    sequence_dropdown,
    widgets.HBox([start_input, end_input, stride_input, focus_input, max_static_input]),
    widgets.HBox([category_text, gaze_scale_dropdown, gaze_length_slider]),
    widgets.HBox([show_static_checkbox, show_dynamic_checkbox, show_centers_checkbox, show_skeleton_checkbox]),
    widgets.HBox([show_skeleton_traj_checkbox, show_head_checkbox, show_gaze_rays_checkbox, show_gaze_points_checkbox]),
    render_button,
])

display(controls, output)
render()

Output()

## Notes

- `max static = 0` means draw all static object boxes. A smaller positive value keeps the first N static boxes and makes the figure lighter.
- `categories` accepts a comma-separated list such as `cup, plate, food object`.
- Dynamic boxes are selected at the object timestamp nearest to the focus frame timestamp.
- `gaze points` are depth-defined gaze points from the gaze export; they are not object-hit points yet.
- Skeleton joints and object boxes are both in ADT Scene/world frame, meters.